# Cache-to-Cache

### A from-scratch reading of direct semantic communication between language models

*Qwen2.5-0.5B-Instruct as Sharer, Qwen3-0.6B as Receiver, on one CPU with 16 GB of memory.*

## Preface

This document reproduces the central mechanism of *Cache-to-Cache: Direct Semantic
Communication Between Large Language Models* (ICLR 2026) from scratch, and then asks what
it costs.

The guiding question is not whether the paper's numbers can be matched. It is whether the
principle behind them can be reproduced and fully understood using models smaller than
most of those in the paper. The phrase *fully understood* carries the weight. A
replication that produces the right numbers without understanding is worth less than a
replication that produces different numbers and explains why.

The configuration here is the smallest one the paper tests, run on a machine with no
discrete accelerator. That is a deliberate choice rather than a limitation to apologise
for. A mechanism whose behaviour is only legible at scale is a mechanism whose behaviour
is not legible. Running at the floor makes the costs visible in a way that abundant
hardware hides, and the costs turn out to be the most interesting part of the result.

Figures are committed images referenced from markdown, so the document reads without being
executed. They are produced in a separate repository and arranged here.

### What this document assumes and does not repeat

The premise that a contextually enriched cache carries value after its source is discarded
was established separately, in the cache enrichment oracle replication, and is cited in
Section 1 rather than re-derived. The internals of rotary position encoding, grouped-query
attention, and query-key normalisation each have their own reference document; this
notebook uses them and explains only the parts that bear directly on transferring a cache
between two models.

What is not assumed is the mechanism itself. Every component of the fuser is derived here
from the shapes it has to reconcile, because a component accepted on the paper's authority
is a component that cannot be debugged when it misbehaves.

### The shape of the argument

Section 1 states the premise and prices the channel. Section 2 establishes what the two
caches actually are and why they cannot be exchanged directly. Section 3 builds the
projection, shows that projection alone fails, and diagnoses why. Section 4 adds the
residual formulation and the gate, which is where the mechanism starts working. Section 5
measures the gain and then decomposes it into a part the benchmark reads and a part it
discards. Section 6 prices the whole exchange rather than the part that improves, and that
is where the paper's efficiency argument inverts. Section 7 says what follows.

### How to read a figure here

Every figure and equation is followed by the same three movements, in the same order.

1. **Point.** One sentence naming one specific feature inside the artefact.
2. **Claim.** What that feature establishes.
3. **Limit.** What it does not establish, or what would overturn it.

The third movement is the one that matters. An artefact for which no limit can be written
is an illustration rather than evidence, and belongs in an appendix.

### One standing rule

Every efficiency claim in this document names the link it assumes. A statement about speed
that does not say what the two models are connected by is true inside one machine and
false outside it, and the difference is the subject of Section 6.

---

## 1. Text is the wire format, and it is the wrong one

When two language models exchange information today, one decodes its internal state into a
sequence of tokens and the other re-encodes that sequence into a state of its own. Text is
the wire.

### 1.1 What serialisation discards

Text is a poor wire for this, and the reason is structural rather than incidental. A model's
reading of an input is a set of high-dimensional vectors, one per position per layer, each
encoding not a word but that word as understood in the presence of every word before it. The
same token at two positions produces two different vectors. That contextual dependence is the
entire content of what the model computed.

Decoding collapses this to a token sequence. A token is a symbol drawn from a fixed
vocabulary, identical wherever it appears, carrying no record of the state that produced it.
What survives is whatever the model chose to say. What does not survive is why it chose to
say it, the alternatives it weighed, and the structural relationships it built between parts
of the input that never appear adjacent in the output string.

The paper's example makes the loss concrete. A token like `<p>` may be represented inside one
model as *the boundary after which content begins*, a piece of structural knowledge built
from exposure to markup. Serialised, it becomes three characters. A receiver without the same
exposure reads three characters. The structural knowledge was never in the string, so no
amount of decoding sophistication recovers it.

There is a second cost, and it is not about fidelity. Serialisation is sequential. A sender
producing `n` tokens pays `n` forward passes, each depending on the last. The latency of a
text exchange is set by the length of the message, and no hardware removes that dependency,
because it is a property of autoregressive decoding rather than of the machine.

### 1.2 The alternative

The alternative is to move the representation itself: take the key-value cache one model has
already computed while reading the context, project it into the representational space of
another model, and combine it with that model's own cache.

This is attractive for two reasons mirroring the two costs. The cache is what the sender
actually computed, so nothing is discarded in the handoff. And the cache already exists after
prefill, a single parallel forward pass over the whole context, so no sequential decode is
needed to produce it.

### 1.3 The premise, established separately

Before building any of that, it is worth knowing whether the thing being transferred has
value. A prior experiment in this project established that it does. When a question's cache
is computed in the presence of exemplars, and the exemplars are then discarded from the cache
before the model answers, accuracy rises from 0.454 to 0.498 against a cold read of the same
question. The paired bootstrap interval is [+0.021, +0.068] and McNemar gives p = 0.0003.

The mechanism is worth stating precisely, because it is easy to mistake for something weaker.
The cache that remains after the exemplars are cut is exactly as long as the cache from
reading the question alone. Every position corresponds to a question token. Nothing was
appended. What differs is only the content of those positions, computed while the exemplars
were visible, carrying that influence forward after their source is gone. Enrichment survives
the removal of its source at no cost in cache length.

That experiment manufactured its own enrichment. The model prefilled the exemplars itself,
paying their full cost at inference time, and the enriched cache never left the model that
produced it. It is a proof that the asset exists, not a proof that it can be moved. Moving it
introduces a problem the oracle never faced: the cache now has to be intelligible to a model
that did not compute it.

### 1.4 The width of the second channel

The two channels can be compared directly, because both carry the Sharer's reading of the
same context and both can be counted in bytes.

The cache path carries, for every position of context, the Sharer's key and value vectors at
every layer. For Qwen2.5-0.5B that is 24 layers, each holding 2 key-value heads of dimension
64, for both keys and values, at two bytes per element in fp16:

```
24 layers x 2 heads x 64 dims x 2 tensors x 2 bytes = 12,288 bytes per position
```

The text path carries a short natural-language message. A single clear sentence describing
the context, which is what the paper's text-to-text baseline asks the Sharer to produce, is
on the order of 128 bytes.

Both numbers describe the same thing: how much of the Sharer's reading reaches the Receiver.

![Channel widths, representation against text](assets/c2c_channel_widths.png)

The representation bar carries 12,288 bytes for every position of context; the text bar
carries roughly 128 bytes for an entire short response.

The gap is the premise stated quantitatively. Whatever the Sharer understood about the
context is present in the first bar and mostly absent from the second, and no decoder
sophisticated enough to recover the difference exists, because the difference was discarded
before the string was written.

This figure counts what could be sent. It does not price sending it. The width that appears
here as available bandwidth is measured on the assumption that moving a tensor between the
two models is free, which is true when they share a memory space and is the subject of
Section 6 when they do not.

---

## 2. The two caches are different objects

Before anything can be transferred, it has to be established what is being transferred
between. The Sharer's cache and the Receiver's cache are not two instances of one type.
They differ in every dimension that matters, and one of the differences is not a difference
of size at all.

### 2.1 What a key-value cache is

A transformer layer computes, for each position, three projections of that position's
hidden state: a query, a key, and a value. Attention at position `t` compares the query at
`t` against the keys at every position up to `t`, turns those comparisons into weights, and
returns the weighted sum of the corresponding values.

The keys and values at earlier positions do not change when a new position arrives. They
depend only on their own hidden states, which are already fixed. So they are computed once
and kept. That store is the key-value cache, and its purpose during generation is to avoid
recomputing the entire prefix for every new token.

For the transfer problem, a different property of the same object is what matters. The
cache is the model's digested reading of the context, held in the form the model's own
attention consumes. It is not the model's output and it is not the model's input. It sits
exactly at the interface where one part of the model hands understanding to another part,
which is what makes it the natural thing to hand to a different model entirely.

A cache has four axes: batch, key-value head, position, and head dimension. The position
axis is the context. The other three are architecture, and they are where the two models
disagree.

### 2.2 Where the cache is taken from

A cache is not simply "the keys". It is the keys at one specific point in a pipeline, and
the two models do not have the same pipeline.

![Where each model's key cache is taken](assets/c2c_schematic_capture_point.png)

The Sharer's column has a dashed box where the Receiver's has `k_norm, per head`.

The capture point sits below rotary embedding on both sides, so whatever crosses is already
position-encoded, and it sits below query-key normalisation on the Receiver's side only,
because Qwen2.5 has no such step. The tensor that crosses therefore differs from its
destination not only in width but in what has already been done to it. A Sharer key has
been projected and rotated; a Receiver key has been projected, normalised per head, and
rotated. The fuser is the only thing standing between those two histories.

The schematic shows where the tensors are taken, not what that difference costs. Whether
the missing normalisation step matters for reconstruction is not something a pipeline
diagram can answer, and this project did not isolate it.

### 2.3 Where the two models disagree

| | Sharer, Qwen2.5-0.5B-Instruct | Receiver, Qwen3-0.6B |
|---|---|---|
| Layers | 24 | 28 |
| Hidden size | 896 | 1024 |
| Query heads | 14 | 16 |
| Key-value heads | 2 | 8 |
| Head dimension | 64 | 128 |
| Cache width per layer, per position | 2 x 64 = 128 | 8 x 128 = 1024 |
| Cache bytes per position, fp16 | 12,288 | 114,688 |

The bottom two rows are the transfer problem in miniature. A Sharer key vector for one
position at one layer is 128 numbers. The Receiver key vector it would have to stand in for
is 1024 numbers. No reshaping, padding, or reinterpretation turns one into the other,
because the second is not a rearrangement of the first; it is a different quantity of
information about the same text.

Three entries are traps, and each has cost time somewhere in this project or would have.

**Head dimension is not hidden size over head count.** The Receiver's hidden size is 1024
and it has 16 query heads, so the arithmetic gives 64. The actual head dimension is 128.
Qwen3 decouples the two through a fallback that reads as boilerplate. Code computing head
dimension rather than reading it produces tensors of the wrong shape, and because the wrong
shape is a plausible shape, nothing raises.

**Key-value head count is not query head count.** Both models use grouped-query attention.
The Receiver's 16 query heads share 8 key-value heads; the Sharer's 14 share 2. The cache is
stored at the key-value head count and expanded to query width on every forward pass, so a
captured cache is narrower than the attention that consumes it, and anything written back
must match the stored width. Writing at query width raises a shape error if you are lucky
and broadcasts silently if you are not.

**Layer counts differ by four,** and no arithmetic makes 24 into 28.

### 2.4 Choosing a layer correspondence

A fuser operates per layer, so each Receiver layer that receives anything needs a Sharer
layer to receive it from. With 24 and 28, four Receiver layers will have no source.

Two correspondences are defensible. Depth-normalised alignment maps both index ranges onto
[0, 1] and pairs nearest neighbours in that normalised space, giving every Receiver layer a
partner at the cost of reusing some Sharer layers. Terminal alignment pairs from the output
side inward: last to last, penultimate to penultimate, descending until the shallower model
runs out.

This project uses terminal alignment, following the paper. Deep layers hold representations
that are semantic and largely detached from surface form; shallow layers hold
representations still tied to tokens and local syntax. Two different models are most likely
to be talking about the same thing deep in the stack. Anchoring at the output puts the
reliable pairs where alignment is most likely to be meaningful and lets the mismatch fall
where it matters least.

![Terminal alignment between the two stacks](assets/c2c_layer_mapping.png)

The four shallowest Receiver layers have no partner.

The chain runs from Receiver layer 27 taking Sharer layer 23 down to Receiver layer 4
taking Sharer layer 0, giving 24 pairs. Receiver layers 0 through 3 operate on their own
input exactly as they always did, with no fuser attached and nothing injected. This is not
a gap to be patched: those layers are doing the tokenisation-adjacent work a Sharer's deep
layers would have nothing useful to say about, and leaving them alone is the correct
handling rather than a compromise.

The figure shows a mapping, not its justification. Depth-normalised alignment produces a
different mapping this project did not test; the paper reports it as slightly worse and
this notebook takes that on trust rather than on measurement, which makes the choice here
inherited rather than verified.

### 2.5 Tokens have to line up too

Layer alignment answers where a Sharer vector goes. Token alignment answers which Sharer
vector goes there.

The position axis indexes tokens, and two tokenisers reading the same string do not
necessarily produce the same count or the same boundaries. If the Sharer's cache holds 47
positions for a context the Receiver tokenises into 51, there is no position-wise
correspondence to fuse along, and fusing anyway combines a Sharer's reading of one substring
with a Receiver's reading of a different one. The failure is silent: both tensors have valid
shapes along every axis except the one carrying meaning.

The general solution is to decode each Receiver token back to its string form and re-encode
it with the Sharer's tokeniser, keeping the token with maximal string coverage when several
result. Chat-template sections, which carry no semantic content, are padded to equal length
instead.

Whether that machinery is needed for a given pair is an empirical question about those two
tokenisers, answered by comparing them rather than assumed from shared lineage.

### 2.6 Keys and values are not the same kind of object

One more distinction has to be drawn before a projection can be designed, and it is internal
to a single cache rather than between the two models.

![Rotary on the keys, nothing carried on the values](assets/c2c_schematic_key_value_asymmetry.png)

The value column has two dashed boxes where the key column has `k_norm` and
`rotary embedding`.

Only one of the two carries position. A key's geometry encodes where it was computed, so
moving it to a different position changes what it means; a value has no such dependence.
Keys and values sit side by side in the same structure and are, for transfer purposes,
different kinds of object.

The diagram is the architecture, not a measurement. That the paths differ on paper does not
establish that the difference is visible in the tensors, which is what the next figure
checks.

![Rotary ladders in keys, absent in values](assets/c2c_rope_ladders.png)

The key curves separate into a ladder ordered by feature dimension; the value curves do not
separate at all.

The architectural asymmetry is present in the tensors. Low feature dimensions rotate slowly
and encode coarse positional structure; high dimensions rotate quickly and encode fine
structure. The absence of any such ordering among the value curves confirms values were left
untouched.

This shows the two tensor types differ, not that the difference should change the projection
design. A prediction that value projections would therefore reach lower relative
reconstruction floors than key projections, on the reasoning that values are the simpler
object, was written before the measurement and was falsified across all 24 paired layers.
The geometric distinction is real and its consequence for reconstruction difficulty runs the
other way.

### 2.7 What the two representation spaces look like

The shapes disagree, which is a fact about the architectures. Whether the contents disagree
is a fact about the training, and it decides whether a mapping between them can exist at
all.

![The two representation substrates](assets/c2c_substrate_triptych.png)

The Sharer and Receiver panels occupy visibly different regions, and neither is a scaled
copy of the other.

Two models from the same family, trained on overlapping data, reading identical text, encode
it into distinct representational territories. This is simultaneously what makes the
transfer non-trivial and what makes it worth attempting. If the regions coincided there
would be nothing to gain, because the Receiver would already hold what the Sharer holds. If
they were unrelated there would be nothing to map, because no function could carry one to
the other without inventing the difference.

Occupying different regions is not the same as encoding different information. The panels
describe where representations sit, not what they know, and a difference in position is
compatible with complete redundancy in content. Section 3.4 settles that question, and the
answer is not the encouraging one.

### 2.8 What failure looks like

A projection that has learned nothing still produces output, and that output still has a
reconstruction error. Knowing what that error is prevents mistaking a converged trivial
solution for a working map.

Mean-squared reconstruction has a closed-form trivial solution: predict the target
distribution's mean for every input, ignoring the input entirely. That solution achieves a
loss equal to the target's total variance, meaningfully lower than a random initialisation,
and therefore produces a curve that descends and flattens exactly the way a successful run
does. Normalising by it puts the null at 1.0, which is the scale every reconstruction number
in this document uses.

![The geometric null](assets/c2c_geometric_null.png)

The null band marks the reconstruction error achieved by a mapping that has learned nothing
about its input.

Any projection has to clear this band before its output can be called a projection at all.
Without the band, a loss curve that descends and flattens is indistinguishable from a loss
curve that has found the mean, and every claim built on that curve inherits the ambiguity.

The null establishes a floor for the reconstruction objective only. Clearing it says the
projection has learned something about the input. It says nothing about whether what it
learned is the part the Receiver needs, which is the distinction the next section is built
around.

---

## 3. A projection alone does not carry the meaning

Section 2 established the mismatch. This section builds something to bridge it, then
establishes that bridging the shapes is not the same as bridging the meaning.

### 3.1 Building the projection

The requirement is a function taking a Sharer cache entry and producing something
Receiver-shaped. Per layer, per position, for keys:

```
Sharer key      [batch, 2 heads, T, 64 dims]    ->  flatten heads  ->  [batch, T, 128]
Receiver key    [batch, 8 heads, T, 128 dims]   ->  flatten heads  ->  [batch, T, 1024]
```

The projection concatenates the two along the feature axis and maps the result to Receiver
width:

```
concat          [batch, T, 128 + 1024]  =  [batch, T, 1152]
hidden          [batch, T, 256]
output          [batch, T, 1024]        ->  unflatten  ->  [batch, 8, T, 128]
```

Values follow the same path with their own parameters. Two design points are worth stating
rather than reading past.

**The Receiver's own cache is an input, not just a target.** The projection could have been
a pure Sharer-to-Receiver map, 128 numbers to 1024. It takes 1152 to 1024 instead, so it can
condition what it writes on what the Receiver already computed. This is the difference
between translating a message and answering one, and it is the first place the design
assumes the two caches are complementary rather than redundant.

**The hidden layer is 256 wide,** narrower than either the input or the output. Section 3.6
returns to whether that bottleneck is the binding constraint. It is not.

### 3.2 Substitute or add

With a projection in hand there are two ways to use it. Its output can replace what the
Receiver computed, or it can be added to it.

The difference is one flag and a large difference in what is being claimed. Replacement
asserts that a projection of the Sharer's cache is a sufficient cache for the Receiver.
Addition asserts only that the Sharer's reading contains something the Receiver's own
reading lacks. The first is much stronger, and it is worth testing first, because if it holds
then the machinery of Section 4 is unnecessary complexity.

### 3.3 The projections clear their own objective

Each of the 24 paired layers gets one projection for keys and one for values, each trained to
reconstruct the Receiver's tensor from the Sharer's. Loss is reported relative to the null of
Section 2.8, so 1.0 is the score of a map that ignores its input.

![Held out loss relative to the null, per layer](assets/c2c_loss_vs_null.png)

Every one of the 48 curves ends below the dashed line at 1.0.

The projections learn. Keys settle roughly between 0.34 and 0.66 and values between 0.50 and
0.88, which also settles the prediction recorded in Section 2.6: keys reach lower floors than
values in all 24 layers, the opposite of what the simpler geometry of values suggested. A
trained map beats an untrained one everywhere.

Beating the null is a low bar. The best layer still carries a third of the null's error and
the worst carries seven eighths, and nothing in these curves says whether what survives
reconstruction is the part the Receiver needs.

### 3.4 Where the projected cache lands

![Where the projected cache lands](assets/c2c_distribution_shift.png)

The projected distribution sits inside the target region and occupies a proper subset of it.

The projection is not producing noise and not producing the mean; it moves Sharer
representations into the territory Receiver representations inhabit. What it does not do is
cover that territory. The Receiver's cache spans directions the projected cache never
reaches, which makes a projected cache a Receiver-shaped object missing part of what
Receiver-shaped objects normally contain.

Subset occupancy is measured in the space this probe projects into, and a subset in a
low-dimensional view is not necessarily a subset in the full space. The claim survives
because the missing directions also show up in the reconstruction floors above, but this
figure alone would not establish it.

### 3.5 Two ways of answering nothing

Reconstruction quality and usefulness are different questions, and the second is answered by
letting the Receiver decode from what the projection produced. Two of the four builds
evaluated in this project do exactly that: one substitutes the MSE-trained projection
directly, and one substitutes the output of a fuser trained end to end. If a projection alone
were enough, either would answer roughly the way the questions are distributed.

![What each build answered](assets/c2c_answer_distributions.png)

Two of the four builds put almost every answer on a single letter: the MSE projection answers
A 498 times out of 500, and the substituting build answers B 497 times.

Neither model was handed a worse reading of the question. Each was handed no usable reading
at all, and what remains is an unconditioned preference for one letter. That the two
collapses land on different letters is the clearest evidence that neither letter is about the
question; each is an artefact of whatever survived its own substitution. The total variation
distances against the answer key say it numerically, 0.800 and 0.744, against 0.336 for the
Receiver alone and 0.118 for the adding build.

The figure does not show that no projection could carry this information. It shows that these
two, trained under these objectives, did not. A better-trained projection substituted the
same way would be a different experiment, not a refutation of this one.

### 3.6 Why the projection cannot reach the target

The obvious explanation for those collapses is capacity. The 256-wide hidden layer is
narrower than the 1024-wide target, and a bottleneck narrower than the signal is an obvious
suspect.

The suspicion is testable. Compress the target to 256 dimensions optimally, using its own
principal components, and reconstruct it. That is the best any architecture with this
bottleneck could achieve regardless of training, and it gives a relative loss of 0.146. The
worst layer in Section 3.3 sits at 0.880. The bottleneck accounts for roughly a sixth of that
gap and the remaining five sixths are something else.

The something else is informational. If the target's variance were concentrated in a few
directions, a wider layer would capture them and the floor would fall. It is not. The leading
two principal directions hold 10.89% of the variance, and the participation ratio, which
counts how many directions carry comparable weight, is 82.6. The variance is spread across
roughly eighty directions of similar magnitude. There is no low-dimensional structure to
widen into.

Stated plainly: the Sharer's cache does not contain a compressible image of the Receiver's
cache. What limits reconstruction is what the Sharer knows about this context, not how much
of it fits through the pipe. Section 2.7 left open whether two different representational
regions meant two different contents. This is the answer. They do.

This is the more useful failure. An architectural limit invites a bigger model, a fix
available to anyone with more hardware and unavailable here. An informational limit says the
objective is wrong, and the objective is free to change.

The objective in question asks the projection to predict what the Receiver would have
computed. That target is unreachable and, on reflection, was never the right one. Nothing
about the task requires reproducing the Receiver's cache. What the task requires is producing
a cache the Receiver can use, which is both weaker and better aligned with the outcome anyone
cares about.

---

## 4. Residual and gate are what make it work

Two changes follow from Section 3. The fused cache keeps the Receiver's own contribution
rather than discarding it, and the objective becomes the Receiver's own next-token loss
rather than a reconstruction target.

### 4.1 The residual formulation

For each Receiver layer `n` that has an aligned Sharer layer `G(n)`:

$$C^{F}_{n} = C_{n}(X) \;+\; g_{n} \cdot F_{n}\big(C_{n}(X),\; C^{S}_{G(n)}(X)\big)$$

`C_n(X)` is the Receiver's own cache at that layer, `F_n` is the fuser of Section 3.1, and
`g_n` is a per-layer gate. Keys and values have separate fusers and separate gates, so a
layer can open for one and stay shut for the other.

Setting every `g_n` to zero reduces the expression to `C_n(X)`, which is the Receiver
running exactly as it does with none of this machinery attached.

That is not a remark about the algebra. It is a test with a pass condition, and it is the
single most important test in the project. A shut fuser must be bit-for-bit identical to the
unmodified Receiver, evaluated against the real model pair rather than a mock. If it is not,
then something in the injection path is modifying the cache outside the fuser's control, and
every number downstream is describing that modification rather than the mechanism. This test
passes here. Its companion, that gradient actually reaches the fuser's parameters, passes as
well; a fuser correctly wired but disconnected from the loss produces a descending training
curve for reasons that have nothing to do with fusion.

The equation says nothing about whether the correction helps. It guarantees only that the
mechanism can be switched off cleanly, which is what makes the comparison between arms
meaningful without making either arm succeed.

### 4.2 Why the residual changes the problem

Under substitution, the projection had to be right. Anything it failed to reproduce was
lost, because nothing else remained to supply it. Under addition, the projection only has to
be useful. The Receiver's reading of the context is present regardless, so the fuser's
output is a correction to a reading rather than a replacement for one.

This is why the informational limit of Section 3.4 stops being fatal. The Sharer's cache
does not contain a compressible image of the Receiver's cache, and under the residual
formulation it does not need to. It needs to contain something the Receiver's cache lacks,
which is a much weaker requirement and precisely the one that Section 2.6's separated
regions made plausible in the first place.

### 4.3 Why there is a gate at all

A fuser that always fires assumes every aligned layer benefits from fusion. The paper's own
enrichment analysis says otherwise: enriching the best few layers outperforms enriching all
layers, and enriching the worst few is actively harmful. Some layers are places where a
second model's reading helps, and some are places where it interferes.

The gate makes that selection learnable rather than hand-chosen. But a selection is a
discrete thing, and discrete things do not have gradients.

The resolution is a Gumbel-sigmoid with an annealing temperature. Early in training the gate
is a soft value in the open interval, differentiable, so gradients flow through it and the
model can discover which layers are worth opening. The temperature descends linearly from
1.0 to 0.001 across training, sharpening the sigmoid until the soft value is pinned to one
end or the other. At inference the gate is exactly binary: a layer either uses its fused
cache or does not, with no intermediate state to explain and no stochasticity in the
deployed model.

![Gate values across training](assets/c2c_gate_anneal.gif)

The gate values begin spread across the interval and end pinned at zero or one.

The annealing does what it was designed to do. What starts as a continuous weight, carrying
gradient and permitting exploration, ends as a decision. The same parameter serves both
roles across training, which is the point of the construction: there is no separate
discretisation step at the end that could disagree with what training converged to.

The animation shows the gates converging, not that they converged to the right layers. A
gate pattern that is confidently wrong looks identical at this resolution to one that is
confidently right, and nothing here distinguishes them.

### 4.4 How training actually runs

Both models are frozen. The only trainable parameters are the fusers and the gates, which is
what makes 250 steps on a CPU tractable at all.

A single step runs three phases. Both models prefill the context, producing their respective
caches. The fuser combines them under the equation above, and the result replaces the
Receiver's cache. The Receiver then prefills the target response against the fused cache, and
standard next-token loss on that response backpropagates through the fuser.

The structure explains why the fuser only runs during prefill. Once the fused cache exists,
generation proceeds normally: each new token appends its own key and value, computed by the
Receiver from its own hidden state, with no Sharer involvement. The Sharer contributes to the
context and to nothing after it. Whatever influence it has on the answer travels entirely
through the state the Receiver starts decoding from.

Both arms of Section 3.2 are trained this way, identically, differing only in the residual
flag. Whether that difference is worth anything is Section 5.

---

## 5. The gain is real and smaller than it looks

Four builds answer the same 500 MMLU-Redux questions. The Receiver alone scores 0.382. The
MSE projection substituted directly scores 0.198, below the 0.250 a coin would reach on four
options. The substituting fuser scores 0.250 exactly. The adding fuser scores 0.458.

### 5.1 Testing the differences

An unpaired comparison would waste most of the available information, since every build
answers the same questions. Two paired tests are used, and they answer different questions.

The paired bootstrap resamples questions with replacement and recomputes the difference,
building a distribution of the gap rather than of either rate. It answers whether the gap
depends on which questions happened to be sampled.

McNemar's test looks only at discordant pairs, meaning questions where exactly one build is
correct. It answers whether one build's wins outnumber its losses by more than chance, which
matters because aggregate accuracy can rise while a build breaks as many questions as it
fixes.

![Accuracy under paired statistics](assets/c2c_accuracy_interval.png)

The second row, adding against the Receiver alone, is the only one of the four whose
distribution sits entirely to the right of zero and is also small.

Every comparison here excludes zero, which makes the ordering of the four builds a real
ordering rather than an artefact of sampling. Adding beats the Receiver alone by +0.076,
interval [+0.038, +0.112], McNemar p = 7.66e-05 on 90 discordant pairs. Substituting loses to
it by -0.132, and the MSE projection loses by -0.184 with p = 3.60e-14 on 154 discordant
pairs. The two ways of substituting are not merely worse than adding; they are worse than
doing nothing.

The size of the winning margin deserves attention next to the size of the losing ones. The
gap between adding and substituting is +0.208, nearly three times the gap between adding and
the baseline. Most of what this project demonstrates is the cost of the wrong formulation
rather than the benefit of the right one.

None of these tests says the gain comes from semantic transfer. All are equally compatible
with a fuser that improved the Receiver's output format, its calibration, or its willingness
to commit to an answer at all. Distinguishing those from comprehension requires decomposing
the loss rather than counting correct answers.

### 5.2 What the loss improvement is made of

The next-token loss the fuser trains against is defined over the entire vocabulary. The
grader is not. It reads the probabilities over four answer letters, normalises them to sum to
one, and takes the largest.

That normalisation is the crux. A change in how much probability sits on letters rather than
on everything else is invisible to the grader, because the normalisation divides it out. Only
the relative ordering within the letters survives.

The loss improvement therefore splits into two terms. One is mass moving between answer
letters, which can change an answer. The other is mass moving from non-letters into letters,
which changes only how committed the model is to answering in the required format. Both arms
of Section 3.2 can be decomposed this way, and the comparison is the point.

![Loss improvement split by letter mass](assets/c2c_loss_decomposition.png)

Read from the total alone both arms improved, by -0.517 and -0.200, and the two totals are
made of opposite things.

For the adding arm, -0.393 of the -0.517 is the term the grader discards, which is 76% of the
improvement. For the substituting arm the answer-changing term moved the wrong way, +0.220,
and the total is negative only because the discarded term more than covers it. This is what a
collapse looks like from inside the loss: a build that learned to emit a letter and unlearned
which letter, scoring an improvement on the objective it was trained on while getting worse
at the task.

The caution is that neither answer-changing term is separable from zero. The adding arm's is
-0.123 with an interval of [-0.277, +0.022] on 64 samples, which crosses zero. The part of
the improvement capable of changing an answer is real enough to show up as +0.076 of
accuracy, and this decomposition does not have the resolution to confirm it independently.
The finding that survives is about the substituting arm, whose failure the total alone would
have hidden entirely.

---

## 6. The price, and the link it assumes

The paper's efficiency argument is that cache communication avoids the Sharer's sequential
decode, and it is correct. This section prices the whole exchange rather than the part that
improves.

### 6.1 The cost that is small

Running the fuser across all 24 aligned layers during the Receiver's prefill adds roughly 8%
to that prefill. This is cheap for a structural reason rather than by tuning: the fuser is a
small feed-forward map applied once per position per layer, against a prefill already doing
attention over the whole context at every layer. The fuser's cost is linear in context length
where attention's is quadratic, so the ratio improves as contexts grow.

### 6.2 The cost that depends on the wire

The Sharer's cache has to reach the Receiver, and Section 1.4 counted what that means:
12,288 bytes per position of context, against roughly 128 bytes for a whole short text
message. Per position the ratio is about 96 to 1, and because the cache scales with context
while the message does not, it widens with every token of prompt.

Whether that matters is a question about the link, and it can be written as a condition
rather than a verdict:

```
cache path wins  <=>  payload_bytes / link_bandwidth  <  sharer_decode_time
```

Inside one machine the left side is effectively zero. The two models share an address space,
the cache is already resident after prefill, and handing it over costs a pointer. Across a
link, the condition becomes a threshold with a number attached.

![Break-even link speed](assets/c2c_ledger.png)

The four measured points sit at 8, 15, 31, and 61 Mbit/s as prompt length goes from 64 to
512 positions, at 16 tokens decoded.

Those thresholds are lower than the payload sizes suggest, and lower than this document's
own framing up to this point would predict. Gigabit ethernet clears the worst of them by more
than an order of magnitude, and ordinary wireless clears them too. The mechanism is not
confined to a memory bus. What the surface does show is a gradient: the threshold roughly
doubles as the prompt doubles, so the mechanism gets harder to justify exactly as contexts
grow, which is the opposite of how Section 6.1's compute cost behaves. And the hatched
corner marks a region where fusion costs more than the decode it replaces, so no link speed
rescues it.

The figure prices transport against decode under stated omissions, and its own caption is
explicit that both omissions push the threshold up rather than down: per-token decode is held
at its 16-token value although it grows with context, and the Receiver's extra prefill under
the text pathway is not charged. Nothing is extrapolated past the measured prompt lengths.
These are CPU numbers on a workstation, and the paper's A100 figures are not comparable to
them.

### 6.3 The same number, read twice

Section 1.4 introduced 12,288 bytes per position as the width of the representation channel
and treated it as the reason to prefer that channel: it is how much the Sharer knows that
text cannot carry. The limit written under that figure was that it counted what could be sent
without pricing the sending.

This section is that price, and it is the identical quantity. What was bandwidth in Section 1
is payload here. The property that makes cache communication worth attempting is the same
property that makes it expensive, because the richness and the size are one fact seen from
two ends of the link.

The honest reading of the ledger is narrower than that symmetry suggests, and better. At
these context lengths the price is affordable on ordinary hardware. What the ledger rules out
is not networked cache transfer in general but two specific regimes: links in the
sub-megabit class, and short prompts with short answers, where the fusion compute alone
already exceeds the decode it was meant to replace. Both are recognisable. Neither is where
most deployments sit, and one of them is exactly where the constrained-hardware question of
Section 7 lives.

---

## 7. What the replication settles

The compass question was whether the principle of semantic enrichment through cache transfer
could be reproduced and fully understood using models smaller than those in the paper. It
can. The mechanism works at the paper's smallest configuration on a machine with no
accelerator: adding beats the Receiver alone by +0.076 under two paired tests, the gates
anneal to a clean binary decision, and a shut fuser is exactly a no-op against the real pair.

Understanding it fully turned out to include four things the headline number does not
contain.

**Substitution fails twice, in two different ways, and both are more instructive than the
success.** The MSE projection collapses onto A and the substituting fuser collapses onto B.
Different training, different letter, same shape of failure: a model with no usable reading
of the question falling back on an unconditioned preference. The Receiver's own cache is not
a starting point to be improved but a thing the fused cache must continue to be. The gap
between adding and substituting is +0.208, nearly three times the gap that adding wins over
the baseline, so most of what this project measured is the cost of the wrong formulation.

**The reconstruction limit is informational rather than architectural.** All 48 projections
clear the null, keys reaching lower floors than values in all 24 layers, which falsifies the
prediction written before the run. But a sixth of the residual gap is the bottleneck's fault
and the rest is the absence of any compressible image of one model's cache inside the other's.
Widening the projection recovers the sixth. This is why the objective had to change rather
than the capacity, and it is the step a replication with abundant hardware might have missed
by enlarging the network until something worked.

**Most of what the fuser learns is invisible to the benchmark that motivated it.** Three
quarters of the adding arm's loss improvement lives in a term the grader normalises away. The
same decomposition is what exposes the substituting arm: its answer-changing term moved the
wrong way while its total improved, which the total alone would have hidden. The adding arm's
own answer-changing term does not separate from zero at this sample size, so the accuracy
gain stands on the accuracy measurement rather than on this decomposition confirming it.

**The efficiency argument survives, with a stated link.** Break-even runs from 8 Mbit/s at 64
prompt positions to 61 Mbit/s at 512. That is affordable, and it was not the answer this
document expected while writing Section 1.

### The exchange rate

The ledger is the part that generalises past this pair of models. Fusion costs 8% of a
prefill; transport costs about ninety-six times the bytes of the text it replaces, per
position, and the break-even threshold roughly doubles as the prompt doubles. The compute
cost improves with context and the transport cost worsens with it, so the two halves of the
ledger point in opposite directions and the crossing point moves.

That is a more useful conclusion than either "text is a bad wire format" or "caches are too
big to send". Both are true and neither is decisive. What decides it is a threshold that can
be computed for a given link and a given prompt length, and the contribution of running this
at the smallest scale on the slowest hardware is that the threshold came out as a number
rather than an intuition.

The regime the ledger actually rules out is worth naming precisely, because it is where this
project goes next: links in the sub-megabit class, and short prompts with short answers where
the fusion compute alone exceeds the decode it replaces. A mechanism this expensive to
transmit either shrinks substantially or finds a link that already exists for other reasons,
and which of those two is available is a hardware question rather than a modelling one.

---

## Appendix A. Rotary pairing convention

This is a correctness precondition rather than a result, and it is recorded because getting
it wrong produces no error.

Rotary position encoding rotates pairs of feature dimensions. Which dimensions are paired is
a convention, and two conventions are in circulation. The paper's formulation pairs adjacent
dimensions `(2i, 2i+1)`. The HuggingFace implementation pairs split halves, matching
dimension `i` with dimension `i + d/2`, which is what its `rotate_half` helper computes.

The two are equivalent under an explicit permutation of the feature axis and silently
incompatible without one. A cache produced under one convention and consumed under the other
has correct shapes, correct dtypes, raises nothing, and computes attention scores that are
quietly wrong. Every downstream number would then be a measurement of that corruption.

![Rotary pairing conventions](assets/c2c_rotary_parity.png)

The two conventions agree only after the permutation is applied.

Both models here run under the HuggingFace convention, so the permutation never has to be
applied inside the pipeline. The probe exists to establish that rather than assume it,
because the failure mode it guards against is invisible to any test checking shapes, types,
or finiteness.

The probe verifies the convention for this pair of models under this framework version. It
does not generalise to a model whose implementation differs, and it should be re-run rather
than cited when the substrate changes.

## Appendix B. Environment and framework details

| Item | Detail |
|---|---|
| Sharer | Qwen2.5-0.5B-Instruct, fp16 |
| Receiver | Qwen3-0.6B, fp16 |
| Framework | `transformers` 5.14.1 |
| Cache API | `DynamicCache`, accessed through `.layers[i].keys` and `.layers[i].values` |
| Attention backend | `eager`, required for attention weights to be materialised |
| Hardware | CPU and integrated graphics, 16 GB memory |
| Training data | MMLU auxiliary train split, 250 steps per arm |
| Evaluation | MMLU-Redux, 500 questions, greedy decoding |
| Gate | Gumbel-sigmoid, temperature annealed linearly from 1.0 to 0.001, exactly binary at inference |

Three framework details are load-bearing.

**The cache is stored before head expansion.** `past_key_values.update()` is called before
`repeat_kv` runs, so the stored tensors hold 8 key-value heads for the Receiver, not the 16
query heads that attention consumes. Expansion happens on every forward call. Anything
written into the cache must match the stored width.

**`rope_theta` moved.** The attribute was removed in `transformers` 5.14.1 and its value now
lives in `rope_parameters`. Code reading the old attribute does not fail; it falls back to a
class default, which is a live bug risk in any script carried forward from an earlier
version.

**Head dimension is read, never computed.** As Section 2.2 notes, `hidden_size //
num_attention_heads` gives 64 for the Receiver where the true value is 128.

## Appendix C. Where each claim is produced

| Claim | Source |
|---|---|
| Enrichment survives removal of its source | Separate document, cache enrichment oracle |
| Terminal alignment mapping | `align_layers` in the alignment module |
| Shut fuser is a no-op against the real pair | Fuser test suite |
| Gradient reaches the fuser | Fuser test suite |
| Fused and replace arm training | Two runs, identical but for the residual flag |
| Accuracy interval and McNemar | Evaluation harness over 500 questions |
| Reconstruction floor decomposition | Principal component analysis over the target cache |
| Loss decomposition | Letter-mass probe over the evaluation set |
| Fusion cost and payload ledger | Ledger measurement |

Figures are generated separately and committed as files. This notebook arranges them and
does not produce them.